# 4K Master Upscaler — Real-ESRGAN ×4 + GFPGAN + RIFE + ProRes 4444

Upgrades rendered clips to **3840×2160 UHD 4K** using Real-ESRGAN ×4 neural upscale, then runs **RIFE optical-flow interpolation to 24fps**, applies a **per-scene cinematic grade**, and emits both an **H.264 review cut** and a **ProRes 4444 XQ 12-bit master** (broadcast-grade, no gradient banding).

**Setup:**
1. Add your `Beholder_Chunk_*.zip` (or `.mp4` files) as a **Dataset** and attach it.
2. Set Accelerator to **GPU T4** (~2-3 min/clip incl. RIFE + ProRes). CPU also works (slower).
3. **Run All**. `hd_out_4k.zip` appears on the **Output** tab with H.264 reviews + `.mov` masters.

| Knob | Default | Effect |
|---|---|---|
| `USE_FACE_SR` | `True` | GFPGAN face super-res before ESRGAN |
| `FACE_WEIGHT` | `0.7` | GFPGAN blend strength (0=subtle, 1=full) |
| `ESRGAN_TILE` | `512` | Patch size — lower = less RAM/VRAM |
| `RIFE_TFPS` | `24` | RIFE optical-flow interpolation target fps (0 = skip) |
| `TEMPORAL_BLEND` | `0.15` | Cross-frame blend weight — kills ESRGAN per-frame shimmer |
| `PRORES_MASTER` | `True` | Emit ProRes 4444 XQ 12-bit `.mov` master alongside H.264 |
| `PER_SCENE_GRADE` | `True` | Per-chapter color grade keyed off clip-name prefix |

**Pipeline:** hqdn3d denoise → GFPGAN face SR → Real-ESRGAN ×4 (3072×2048) → temporal blend (kills shimmer) → RIFE →24fps → per-scene grade → 4K pad → H.264 review + ProRes 4444 XQ master

**Per-chapter grades:** clips are graded by their name prefix (`c01_*`=cold grey, `c02_*`=warm amber, `c03_*`=teal cyan, `c04_*`=bleach-bypass, `c05_*`=hospital green-amber). Edit `SCENE_GRADES` in the code cell to add chapters.

**Fix included:** auto-patches `basicsr/data/degradations.py` for torchvision ≥ 0.16
(replaces removed `functional_tensor` import — fixes the `ModuleNotFoundError` on Kaggle Python 3.12).


In [ ]:
# ============================================================================
# 4K MASTER UPSCALER  --  Real-ESRGAN x4 + GFPGAN + RIFE + ProRes 4444
# Pipeline: hqdn3d -> GFPGAN -> Real-ESRGAN x4 -> temporal blend (kills
#           per-frame shimmer) -> RIFE frame-interp to 24fps -> per-scene
#           cinematic grade -> 4K pad -> H.264 review cut + ProRes 4444 XQ
#           12-bit master (broadcast-grade, no gradient banding).
# 1) Add your Beholder_Chunk_*.zip (or .mp4s) as a Dataset and attach it
# 2) Set Accelerator = GPU T4 for speed (CPU also works, slower)
# 3) Run All -> 4K clips + hd_out_4k.zip appear on Output tab
# ============================================================================
import os, sys, glob, json, shutil, site, subprocess, zipfile
from pathlib import Path

# -------------------- KNOBS --------------------
INPUT             = ""       # leave "" to auto-pick from /kaggle/input
USE_FACE_SR       = True     # GFPGAN face SR before ESRGAN (recommended)
FACE_WEIGHT       = 0.7      # GFPGAN blend strength (0=subtle, 1=full)
ESRGAN_TILE       = 512      # tile size for RAM control (lower = less RAM)
OUT_DIR           = "/kaggle/working/hd_out_4k"
RIFE_TFPS         = 24       # RIFE interpolation target fps (0 = skip)
TEMPORAL_BLEND    = 0.15     # cross-frame blend weight (0=off, 0.1-0.2 sweet)
PRORES_MASTER     = True     # also emit ProRes 4444 XQ 12-bit master
PER_SCENE_GRADE   = True     # per-scene color grade keyed off clip-name prefix
# -----------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
def run(cmd, **kw): return subprocess.run(cmd, check=True, **kw)

# ---- 1) locate input --------------------------------------------------
if not INPUT:
    _zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True),
                   key=os.path.getmtime, reverse=True)
    _mp4s = glob.glob('/kaggle/input/**/*.mp4', recursive=True)
    INPUT = _zips[0] if _zips else ('/kaggle/input' if _mp4s else '')
assert INPUT, 'No .zip or .mp4 found under /kaggle/input -- attach your dataset.'
print(f'[input] {INPUT}')

# ---- 2) unpack zip if needed -----------------------------------------
WORK = '/kaggle/working/_upscale_tmp'; os.makedirs(WORK, exist_ok=True)
if INPUT.endswith('.zip'):
    print('[unzip] extracting...')
    with zipfile.ZipFile(INPUT) as zf: zf.extractall(WORK)
    CLIP_DIR = WORK
else:
    CLIP_DIR = INPUT
clips = sorted(glob.glob(f'{CLIP_DIR}/**/*.mp4', recursive=True))
print(f'[clips] {len(clips)} found')
assert clips, f'No .mp4 files found in {CLIP_DIR}'

# ---- 3) install + patch basicsr --------------------------------------
print('\n[setup] installing gfpgan + realesrgan...')
try:
    import gfpgan, realesrgan
except ImportError:
    run([sys.executable,'-m','pip','install','-q',
         'gfpgan','basicsr','facexlib','realesrgan'])

# Patch basicsr for torchvision >= 0.16
# (functional_tensor module was removed; rgb_to_grayscale moved to .functional)
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    _bad = os.path.join(sp, 'basicsr', 'data', 'degradations.py')
    if os.path.exists(_bad):
        _src = open(_bad).read()
        if 'functional_tensor' in _src:
            _src = _src.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            open(_bad,'w').write(_src)
            print('[patch] basicsr degradations.py fixed for torchvision >= 0.16')
        break

# Flush stale module cache after patch
for _m in [k for k in sys.modules
           if k.startswith(('basicsr','gfpgan','realesrgan'))]:
    sys.modules.pop(_m, None)

import torch, cv2
from gfpgan import GFPGANer
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

USE_GPU = torch.cuda.is_available()
print(f'[device] {"GPU (fp16)" if USE_GPU else "CPU (fp32) -- slow but safe"}')

# ---- 4) load models --------------------------------------------------
if USE_FACE_SR:
    face_model = GFPGANer(
        model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
        upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None,
    )
    print('[model] GFPGAN loaded')

rrdb = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
               num_block=23, num_grow_ch=32, scale=4)
esr_model = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=rrdb,
    tile=ESRGAN_TILE, tile_pad=10, pre_pad=0,
    half=USE_GPU,
)
print('[model] Real-ESRGAN x4 loaded')

# ---- 4b) RIFE frame interpolator (optional, GPU) ---------------------
# rife-ncnn-vulkan ships a single CLI binary that takes a folder of frames
# and emits the interpolated frames in place. Independent of ESRGAN, so we
# can safely add or remove it without breaking the upscale path.
rife_bin = ''
if RIFE_TFPS and RIFE_TFPS > 0:
    try:
        run([sys.executable,'-m','pip','install','-q','rife-ncnn-vulkan'],
            capture_output=True)
        import rife_ncnn_vulkan as _rife
        rife_bin = getattr(_rife, 'bin_path', None) or shutil.which('rife-ncnn-vulkan') or ''
        if not rife_bin:
            # The PyPI wheel exposes a CLI entrypoint; locate it via shutil
            rife_bin = shutil.which('rife-ncnn-vulkan') or ''
        print(f'[model] RIFE {"loaded" if rife_bin else "unavailable (will fall back to minterpolate)"} -> {RIFE_TFPS}fps')
    except Exception as _e:
        print(f'[model] RIFE not installed: {_e!r} (will fall back to ffmpeg minterpolate if needed)')

# ---- 4c) Per-scene grade resolver -----------------------------------
# Each chapter of "The Saga of the Beholder" has its own emotional palette.
# Key off the shot-id prefix (e.g. "c01_..." / "opening_..." / "m03_...")
# so a single run grades every clip appropriate to its scene.
# To add a chapter: extend the SCENE_GRADES dict below with a new prefix.
SCENE_GRADES = {
    # Cold concrete-grey: institutional dread (chapter 1)
    "c01": "curves=r='0/0 0.45/0.41 1/0.91':g='0/0 0.5/0.49 1/0.94':b='0/0.02 0.5/0.53 1/0.97',eq=contrast=1.06:brightness=-0.02:saturation=0.88",
    # Warm amber: golden-hour hope (chapter 2)
    "c02": "curves=r='0/0 0.4/0.46 1/1.0':g='0/0 0.5/0.5 1/0.95':b='0/0.02 0.5/0.45 1/0.85',eq=contrast=1.02:brightness=+0.02:saturation=1.08",
    # Deep teal-cyan: water/void (chapter 3)
    "c03": "curves=r='0/0 0.5/0.42 1/0.85':g='0/0 0.5/0.5 1/0.96':b='0/0.02 0.45/0.55 1/1.0',eq=contrast=1.05:brightness=-0.01:saturation=0.94",
    # Bleach-bypass: stark confession (chapter 4)
    "c04": "curves=r='0/0 0.5/0.5 1/0.97':g='0/0 0.5/0.5 1/0.97':b='0/0 0.5/0.5 1/0.97',eq=contrast=1.20:brightness=-0.03:saturation=0.62",
    # Green-amber: hospital lifebeat (med / closing)
    "c05": "curves=r='0/0 0.5/0.48 1/0.92':g='0/0 0.5/0.52 1/1.0':b='0/0.02 0.5/0.5 1/0.92',eq=contrast=1.04:brightness=-0.01:saturation=0.98",
}
DEFAULT_GRADE = SCENE_GRADES['c01']  # safe fallback

def scene_grade_for(name):
    """Pick the grade that matches the clip's chapter prefix, else default."""
    if not PER_SCENE_GRADE:
        return DEFAULT_GRADE
    key = name.lower()
    for prefix, g in SCENE_GRADES.items():
        if key.startswith(prefix) or prefix in key:
            print(f'  [grade] scene hit {prefix} -> {g[:60]}...')
            return g
    print(f'  [grade] no scene prefix hit -> default cold-grey')
    return DEFAULT_GRADE

# ---- 5) process each clip -------------------------------------------
# Default grade (only used if PER_SCENE_GRADE is off)
BASE_GRADE = (
    "curves="
    "r='0/0 0.45/0.41 1/0.91':"
    "g='0/0 0.5/0.49 1/0.94':"
    "b='0/0.02 0.5/0.53 1/0.97',"
    "eq=contrast=1.06:brightness=-0.02:saturation=0.88"
)

for ci, clip_path in enumerate(clips):
    name   = Path(clip_path).stem
    cwork  = f'{WORK}/{name}'; os.makedirs(cwork, exist_ok=True)
    fden   = f'{cwork}/den';   os.makedirs(fden, exist_ok=True)
    ffsr   = f'{cwork}/fsr';   os.makedirs(ffsr, exist_ok=True)
    fesr   = f'{cwork}/esr';   os.makedirs(fesr, exist_ok=True)
    fblend = f'{cwork}/blend'; os.makedirs(fblend, exist_ok=True)
    fifps  = f'{cwork}/ifps';  os.makedirs(fifps, exist_ok=True)
    out_clip    = f'{OUT_DIR}/{name}_4k.mp4'
    out_master  = f'{OUT_DIR}/{name}_master.mov'

    print(f'\n=== [{ci+1}/{len(clips)}] {name} ===')

    # probe
    pr = json.loads(subprocess.run(
        ['ffprobe','-v','quiet','-print_format','json','-show_streams', clip_path],
        capture_output=True, text=True).stdout)
    vs  = next(s for s in pr['streams'] if s['codec_type']=='video')
    SW, SH = int(vs['width']), int(vs['height'])
    FPS    = vs['r_frame_rate']
    HAS_AUDIO = any(s['codec_type']=='audio' for s in pr['streams'])
    print(f'  {SW}x{SH} @ {FPS}fps | audio={HAS_AUDIO}')

    # step a: hqdn3d denoise
    den_vid = f'{cwork}/den.mp4'
    run(['ffmpeg','-y','-i',clip_path,'-vf','hqdn3d=4:3:6:4',
         '-c:v','libx264','-crf','18','-preset','fast', den_vid],
        capture_output=True)
    run(['ffmpeg','-y','-i',den_vid, f'{fden}/f%05d.png'], capture_output=True)
    frames = sorted(glob.glob(f'{fden}/f*.png'))
    print(f'  denoise done ({len(frames)} frames)')

    # step b: GFPGAN face SR
    if USE_FACE_SR:
        for idx, fp in enumerate(frames):
            img = cv2.imread(fp, cv2.IMREAD_COLOR)
            _, _, res = face_model.enhance(
                img, has_aligned=False, only_center_face=False,
                paste_back=True, weight=FACE_WEIGHT)
            cv2.imwrite(f'{ffsr}/f{idx+1:05d}.png', res if res is not None else img)
        print(f'  GFPGAN done')
        esr_src = ffsr
    else:
        esr_src = fden

    # step c: Real-ESRGAN x4
    esr_frames = sorted(glob.glob(f'{esr_src}/f*.png'))
    for idx, fp in enumerate(esr_frames):
        img = cv2.imread(fp, cv2.IMREAD_COLOR)
        out_img, _ = esr_model.enhance(img, outscale=4)
        cv2.imwrite(f'{fesr}/f{idx+1:05d}.png', out_img)
        if (idx+1) % 10 == 0 or (idx+1) == len(esr_frames):
            print(f'  ESRGAN: {idx+1}/{len(esr_frames)} [{SW*4}x{SH*4}]')

    # step d: temporal-consistency blend (kills Real-ESRGAN per-frame shimmer).
    # Each frame becomes prev*blend + cur*(1-blend). Blend weight is small
    # (0.10-0.20 is the sweet spot); too high = motion-trails, too low = no effect.
    # Uses cv2.addWeighted which is single-threaded C++; ~30 frames/sec on T4.
    if TEMPORAL_BLEND and TEMPORAL_BLEND > 0:
        print(f'  temporal blend (w={TEMPORAL_BLEND})')
        prev = None
        for idx, fp in enumerate(sorted(glob.glob(f'{fesr}/f*.png'))):
            cur = cv2.imread(fp, cv2.IMREAD_COLOR)
            if prev is None:
                out = cur
            else:
                out = cv2.addWeighted(cur, 1.0 - TEMPORAL_BLEND, prev,
                                     TEMPORAL_BLEND, 0)
            cv2.imwrite(f'{fblend}/f{idx+1:05d}.png', out)
            prev = cur
        final_esr_dir = fblend
    else:
        final_esr_dir = fesr

    # step e: RIFE frame interpolation to 24fps (kills the 16fps choppy look).
    # RIFE is the optical-flow interpolator Netflix actually uses -- no tearing
    # like minterpolate/vsbmc. rife-ncnn-vulkan reads a folder -> doubles frames
    # to the target fps. Falls back to ffmpeg minterpolate (mi_mode=blend) if
    # RIFE not installed, so the run always completes a deliverable.
    final_src_dir = final_esr_dir
    final_fps = float(FPS.split('/')[0])/float(FPS.split('/')[1]) if FPS else 16.0
    if RIFE_TFPS and RIFE_TFPS > 0 and final_fps < RIFE_TFPS - 0.5:
        if rife_bin:
            print(f'  RIFE -> {RIFE_TFPS}fps (optical-flow)')
            try:
                run([rife_bin, '-i', final_esr_dir, '-o', fifps,
                     '-t', f'r{RIFE_TFPS}', '-n', 'rife-v4.6'], capture_output=True)
                final_src_dir = fifps
                final_fps = float(RIFE_TFPS)
            except Exception as _e:
                print(f'  RIFE failed: {_e!r} -> fallback minterpolate (blend)')
        if final_src_dir == final_esr_dir:  # RIFE didn't produce frames
            # minterpolate mi_mode=blend is the safe non-tearing fallback
            interp_in = f'{cwork}/ifps_in.mp4'
            run(['ffmpeg','-y','-framerate',FPS,'-i',
                 f'{final_esr_dir}/f%05d.png',
                 '-c:v','libx264','-crf','0','-pix_fmt','yuv420p', interp_in],
                capture_output=True)
            run(['ffmpeg','-y','-i',interp_in,
                 '-vf',f'minterpolate=fps={RIFE_TFPS}:mi_mode=blend',
                 '-c:v','libx264','-crf','0','-pix_fmt','yuv420p',
                 f'{fifps}/src.mp4'], capture_output=True)
            run(['ffmpeg','-y','-i',f'{fifps}/src.mp4',
                 f'{fifps}/f%05d.png'], capture_output=True)
            final_src_dir = fifps
            final_fps = float(RIFE_TFPS)
    else:
        print(f'  target fps already >= {RIFE_TFPS} -> skipping interpolation')

    # step f: choose the per-scene grade (or fall back to the default cold-grey)
    grade = scene_grade_for(name) if PER_SCENE_GRADE else BASE_GRADE

    # step g: H.264 review cut + ProRes 4444 XQ master (12-bit, broadcast-grade)
    # The review cut is small/scrubbable; the .mov master is what the editor /
    # colorist actually grades against. 12-bit kills the gradient banding that
    # 8-bit yuv420p shows on HDR-capable displays.
    VF = (
        f"{grade},"
        f"scale=3840:2160:force_original_aspect_ratio=decrease:flags=lanczos,"
        f"pad=3840:2160:(ow-iw)/2:(oh-ih)/2:black"
    )
    fps_arg = f'{int(final_fps) if final_fps.is_integer() else final_fps:.3f}'

    # H.264 review cut
    ff_cmd = ['ffmpeg','-y','-framerate',fps_arg,'-i',f'{final_src_dir}/f%05d.png']
    if HAS_AUDIO:
        ff_cmd += ['-i', clip_path, '-map','0:v','-map','1:a',
                   '-c:a','aac','-b:a','192k']
    ff_cmd += ['-vf',VF,'-c:v','libx264','-crf','14',
               '-preset','slow','-pix_fmt','yuv420p', out_clip]
    run(ff_cmd, capture_output=True)
    sz = os.path.getsize(out_clip)/1e6
    print(f'  H.264 cut  -> {name}_4k.mp4  ({SW}x{SH} -> 3840x2160, {sz:.1f} MB)')

    # ProRes 4444 XQ master (12-bit, broadcast-grade)
    if PRORES_MASTER:
        mf_cmd = ['ffmpeg','-y','-framerate',fps_arg,'-i',f'{final_src_dir}/f%05d.png']
        if HAS_AUDIO:
            mf_cmd += ['-i', clip_path, '-map','0:v','-map','1:a',
                       '-c:a','pcm_s24le']
        mf_cmd += ['-vf',VF,
                    '-c:v','prores_ks','-profile:v','5',  # 5 = ProRes 4444 XQ
                    '-vendor','apl0','-pix_fmt','yuv444p10le',
                    '-colorspace','bt709','-color_primaries','bt709',
                    '-color_trc','bt709',
                    out_master]
        run(mf_cmd, capture_output=True)
        msz = os.path.getsize(out_master)/1e6
        print(f'  ProRes 4444 XQ master -> {name}_master.mov ({msz:.1f} MB, 12-bit)')

    shutil.rmtree(cwork)

# ---- 6) zip all 4K deliverables (H.264 reviews + ProRes masters) -----
zip_path = '/kaggle/working/hd_out_4k.zip'
with zipfile.ZipFile(zip_path,'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(glob.glob(f'{OUT_DIR}/*.mp4') + glob.glob(f'{OUT_DIR}/*.mov')):
        zf.write(f, Path(f).name)
zip_mb = os.path.getsize(zip_path)/1e6

n_mp4 = len(glob.glob(f'{OUT_DIR}/*.mp4'))
n_mov = len(glob.glob(f'{OUT_DIR}/*.mov'))
print(f'\n[ok]  hd_out_4k.zip  ({zip_mb:.1f} MB)  -> Output tab')
print(f'   {len(clips)} clip(s) processed')
print(f'   {n_mp4} H.264 review cuts + {n_mov} ProRes 4444 XQ masters (12-bit)')
pipeline = 'hqdn3d -> GFPGAN (w=' + str(FACE_WEIGHT) + ') -> Real-ESRGAN x4'
if TEMPORAL_BLEND and TEMPORAL_BLEND > 0:
    pipeline += f' -> temporal blend (w={TEMPORAL_BLEND})'
if RIFE_TFPS and RIFE_TFPS > 0:
    pipeline += f' -> RIFE {RIFE_TFPS}fps'
if PER_SCENE_GRADE:
    pipeline += ' -> per-scene grade'
else:
    pipeline += ' -> uniform grade'
pipeline += ' -> 3840x2160 (H.264 review + ProRes 4444 XQ master)'
print(f'   pipeline: {pipeline}')
shutil.rmtree(WORK)
